# Canada Economic Indicators Capstone (Colab)

**Goal:** Explore how Canadian macroeconomic indicators relate to unemployment and build a baseline model to predict unemployment rate by year.

This notebook is designed to run in **Google Colab** with the dataset stored in `data/canada_world_bank_indicators.csv`.

## Data Source
The dataset contains public World Bank indicators for **Canada** (annual observations).
Source: https://data.worldbank.org (downloaded from a public repository mirroring World Bank data).

Fields include population, birth/death rates, employment by sector, GDP, energy use, and unemployment.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from pathlib import Path

%matplotlib inline

In [ ]:
data_path = Path('data/canada_world_bank_indicators.csv')
if not data_path.exists():
    raise FileNotFoundError("Dataset not found. Ensure the CSV is uploaded to data/canada_world_bank_indicators.csv")

df = pd.read_csv(data_path)
df.head()

In [ ]:
df.info()

## Data Cleaning & Feature Prep

We normalize column names to snake_case. For modeling, we focus on 1990+ years because several indicators are flat in earlier years (likely missing or imputed).

In [ ]:
def to_snake(name):
    name = name.strip().lower()
    name = name.replace('%', ' percent')
    name = name.replace('(', ' ').replace(')', ' ')
    name = name.replace('.', '')
    name = re.sub(r'[^a-z0-9]+', '_', name)
    return name.strip('_')

df = df.copy()
df.columns = [to_snake(col) for col in df.columns]

numeric_cols = df.columns.drop(['country'])
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

missing_summary = df.isna().sum().sort_values(ascending=False)
missing_summary[missing_summary > 0]

In [ ]:
df = df.dropna()
df['gdp_per_capita_usd'] = df['gdp_in_usd'] / df['total_population']
df = df.sort_values('year')

model_df = df[df['year'] >= 1990].copy()
model_df.head()

## Exploratory Analysis

In [ ]:
plt.figure(figsize=(10, 5))
sns.lineplot(data=df, x='year', y='unemployment_percent', marker='o')
plt.title('Canada Unemployment Rate (World Bank)')
plt.ylabel('Unemployment (%)')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.lineplot(data=df, x='year', y='gdp_in_usd', marker='o')
plt.title('Canada GDP (USD)')
plt.ylabel('GDP (USD)')
plt.show()

In [ ]:
corr = model_df.drop(columns=['country']).corr(numeric_only=True)['unemployment_percent'].sort_values(ascending=False)
corr.head(8)

## Baseline Modeling: Predict Unemployment

We compare a simple Linear Regression against a Random Forest. The forest uses 100 trees as a stable baseline and can be tuned later.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

target = 'unemployment_percent'
features = model_df.drop(columns=['country', target, 'gdp_in_usd', 'total_population'])

X_train, X_test, y_train, y_test = train_test_split(
    features, model_df[target], test_size=0.2, random_state=42
)

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = mean_squared_error(y_test, preds, squared=False)
    results.append({'model': name, 'mae': mae, 'rmse': rmse})

results_df = pd.DataFrame(results).sort_values('rmse')
results_df

In [ ]:
best_model_name = results_df.iloc[0]['model']
best_model = models[best_model_name]

if hasattr(best_model, 'feature_importances_'):
    importance = pd.Series(best_model.feature_importances_, index=features.columns)
elif hasattr(best_model, 'coef_'):
    importance = pd.Series(best_model.coef_, index=features.columns).abs()
else:
    raise ValueError(
        f'Selected model {best_model_name} does not expose feature importances or coefficients. '
        'Expected a tree-based or linear model.'
    )

importance.sort_values(ascending=False).head(10)

## Key Takeaways & Limitations
- Canada’s unemployment rate shows noticeable shifts during major macroeconomic swings.
- GDP and employment sector mix appear strongly associated with unemployment in this dataset.

**Limitations:**
- The dataset is annual with relatively few rows, so model performance may be unstable.
- This is a single-country dataset; results may not generalize without more context.
- External factors (policy changes, global shocks) are not explicitly captured.